# DDP GPU runs (Colab)

Runs the parts of the project that need a GPU:

1. **Mixed-precision benchmark**: fp32 vs AMP (float16 + GradScaler) on the same model and batches.
2. **Longer masked pre-training**: a bigger ViT (dim 192, depth 6) for 200 epochs, random masking vs policies B and C.
3. **Label efficiency** of those encoders.

Before you start: run `python make_colab_bundle.py` on the laptop, upload `colab_bundle.zip` to the top of your Google Drive, and choose *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/ddp && mkdir -p /content/ddp && cd /content/ddp && unzip -q /content/drive/MyDrive/colab_bundle.zip
%cd /content/ddp
!ls

## 1. Mixed precision

`images_per_sec` should be clearly higher with AMP on a T4 (tensor cores), and `peak_mem_mb` lower. If `loss_last20` is about the same for both, AMP did not hurt training.

In [ ]:
!python src/amp_benchmark.py --model resnet18 --steps 300
!python src/amp_benchmark.py --model vit --steps 300
!python src/amp_benchmark.py --model cnn --steps 300

## 2. Longer masked pre-training (about 30-60 min per run on a T4)

In [ ]:
for pol in ["random", "B", "C"]:
    !python -W ignore src/train_mae.py --policy {pol} --epochs 200 --dim 192 --depth 6 --heads 3 --amp --seed 0 --tag _big

## 3. Label efficiency of the new encoders

In [ ]:
!THREADS=2 python -W ignore src/label_efficiency.py

## 4. Save results back to Drive

In [ ]:
!zip -qr /content/drive/MyDrive/ddp_gpu_results.zip results checkpoints/mae_*_big_*.pt
print("saved to Drive: ddp_gpu_results.zip")